In [ ]:
# ====================================
# Import
# ====================================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

import joblib

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# ====================================
# Definitions
# ====================================

def evaluate_pr(model, X_test, y_test):

    prob = model.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_test,
        prob
    )

    ap = average_precision_score(
        y_test,
        prob
    )

    print(ap)

    return precision, recall, ap

def plot_pr(baseline, precision, recall, ap):

    plt.plot(
        recall,
        precision,
        label=f"AP = {ap:.2f}"
    )

    plt.axhline(
        y = baseline,
        linestyle="--",
        label="Random baseline"
    )

    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        "Precision-Recall Curve"
    )

    plt.legend()
    plt.show()


# ====================================
# Load
# ====================================

BASE_DIR = Path.cwd()

ini_file = BASE_DIR / "data" / "employees.csv"

df = pd.read_csv(ini_file)

# ====================================
# Data Cleaning
# ====================================

df["PerformanceScore"] = df["PerformanceScore"].fillna(
    df["PerformanceScore"].mode()[0]
)
df["Education"] = df["Education"].fillna(
    df["Education"].mode()[0]
)

# ====================================
# Creating unbalanced data
# ====================================

df["HighSalary"] = (
    df["Salary"] >= df["Salary"].median()
)

df_false = df[
    df["HighSalary"] == False
]

df_true = df[
    df["HighSalary"] == True
]

df_true = df_true.sample(
    n=10,
    random_state=42
)

df = pd.concat(
    [
        df_false,
        df_true
    ]
)

# ====================================
# Feature Selection
# ====================================

X = df[[
    "PerformanceScore",
    "Education",
    "Department",
    "Experience"
]].copy()


# ====================================
# Feature Engineering
# ====================================

X["ExperienceScore"] = (
    X["Experience"] * X["PerformanceScore"]
)

# ====================================
# Target
# ====================================

y = df["HighSalary"]

# ====================================
# Train/Test Split
# ====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================
# Feature split
# ====================================

numeric_features = [
    "PerformanceScore",
    "Experience",
    "ExperienceScore"
]

categorical_features = [
    "Education",
    "Department"
]

# ====================================
# ColumnTransformer
# ====================================

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(drop="first"), 
        categorical_features
    )
])

# ====================================
# Models
# ====================================

logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced"
        )
    )
])

forest_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            max_depth=5,
            n_estimators=100,
            random_state=42
        )
    )
])

svm_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        SVC(
            kernel="linear",
            probability=True
        )
    )
])



# ====================================
# Gradient Model
# ====================================

gradient_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        GradientBoostingClassifier(
            random_state=42
        )
    )
])

gradient_model.fit(
    X_train,
    y_train
)

gradient_pred = gradient_model.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        gradient_pred
    )
)

print(
    confusion_matrix(
        y_test,
        gradient_pred
    )
)

print(
    classification_report(
        y_test,
        gradient_pred
    )
)

# ====================================
# GridSearchCV
# ====================================

param_grid = {
    "model__n_estimators": [
        50,
        100,
        200
    ],
    "model__learning_rate": [
        0.05,
        0.1,
        0.2
    ],
    "model__max_depth": [
        1,
        2,
        3
    ]
}

grid_search = GridSearchCV(
    gradient_model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

best_gradient_model = grid_search.best_estimator_
gradient_pred = best_gradient_model.predict(
    X_test
)

print(
    accuracy_score(
        y_test,
        gradient_pred
    )
)

print(
    confusion_matrix(
        y_test,
        gradient_pred
    )
)

print(
    classification_report(
        y_test,
        gradient_pred
    )
)

print(grid_search.best_params_)
print(grid_search.best_score_)

# ====================================
# Model saving
# ====================================

model_path = BASE_DIR / "models" / "gradient_model.pkl"

joblib.dump(
    gradient_model,
    model_path
)


0.7857142857142857
[[11  2]
 [ 1  0]]
              precision    recall  f1-score   support

       False       0.92      0.85      0.88        13
        True       0.00      0.00      0.00         1

    accuracy                           0.79        14
   macro avg       0.46      0.42      0.44        14
weighted avg       0.85      0.79      0.82        14

0.9285714285714286
[[13  0]
 [ 1  0]]
              precision    recall  f1-score   support

       False       0.93      1.00      0.96        13
        True       0.00      0.00      0.00         1

    accuracy                           0.93        14
   macro avg       0.46      0.50      0.48        14
weighted avg       0.86      0.93      0.89        14



c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

['c:\\MyProject\\PythonApp\\24_gradient_boosting\\models\\gradient_model.pkl']